# Data Cleaning and Standardisation

## Purpose

This notebook cleans and standardises the four AIHW emergency department data tables based on the issues identified during data profiling.

The raw workbook remains unchanged. Cleaning decisions preserve publication-status codes, data-quality flags and structural missingness wherever relevant. 

In [307]:
from pathlib import Path

import numpy as np
import pandas as pd

In [308]:
file_path = Path(
    "../data/raw/myhosp-emergency-department-data-extract.xlsx"
)

In [309]:
# Load individual dataset and use the header defined in profiling
presentations = pd.read_excel(
    file_path,
    sheet_name="Presentations",
    header=15
)

seen_on_time = pd.read_excel(
    file_path,
    sheet_name="Patients seen on time",
    header=16
)

within_4hrs = pd.read_excel(
    file_path,
    sheet_name="Time in ED - within 4 hrs",
    header=19   
)

time_in_ed = pd.read_excel(
    file_path,
    sheet_name="Time in ED",
    header=16
)

In [310]:
print("Presentations:", presentations.shape)
print("Patients seen on time:", seen_on_time.shape)
print("Within 4 hrs:", within_4hrs.shape)
print("Time in ED:", time_in_ed.shape)

Presentations: (22678, 6)
Patients seen on time: (21972, 11)
Within 4 hrs: (36289, 11)
Time in ED: (13611, 13)


In [311]:
presentations_clean = presentations.copy()
seen_on_time_clean = seen_on_time.copy()
within_4hrs_clean = within_4hrs.copy()
time_in_ed_clean = time_in_ed.copy()

In [312]:
# Standardise column names - presentations_clean
presentations_clean.columns = (
    presentations_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

presentations_clean.columns

Index(['reporting_unit', 'reporting_unit_type', 'state', 'year',
       'triage_category', 'number_of_presentations'],
      dtype='str')

In [313]:
# Trim whitespace from categorical columns
categorical_cols = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "year",
    "triage_category",
]

for col in categorical_cols:
    presentations_clean[col] = (
        presentations_clean[col]
        .astype("string")
        .str.strip()
    )

In [314]:
# Standardise state abbreviations 
state_mapping = {
    "NSW": "NSW",
    "Vic": "VIC",
    "Qld": "QLD",
    "WA": "WA",
    "SA": "SA",
    "NT": "NT",
    "Tas": "TAS",
    "ACT": "ACT",
    "NAT": "NAT",
}

presentations_clean["state"] = (
    presentations_clean["state"]
    .map(state_mapping)
)


In [315]:
presentations_clean["state"].value_counts(dropna=False)

state
NSW    12170
VIC     3803
QLD     2370
WA      1720
SA      1460
NT       510
TAS      385
ACT      190
NAT       70
Name: count, dtype: int64

In [316]:
# Preserve small-count suppression
presentations_clean["small_count_flag"] = (
    presentations_clean["number_of_presentations"] == "<5"
)

presentations_clean["number_of_presentations"] = pd.to_numeric(
    presentations_clean["number_of_presentations"],
    errors="coerce"
)

In [317]:
# Validation
print("Rows:", len(presentations_clean))

print(
    "Small-count records:",
    presentations_clean["small_count_flag"].sum()
)

print(
    "Missing numeric presentation counts:",
    presentations_clean["number_of_presentations"].isna().sum()
)

print(
    "Duplicate rows:",
    presentations_clean.duplicated().sum()
)

Rows: 22678
Small-count records: 616
Missing numeric presentation counts: 616
Duplicate rows: 0


In [318]:
# Validate key
candidate_key = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "year",
    "triage_category",
]

assert presentations_clean.duplicated(
    subset=candidate_key
).sum() == 0

In [319]:
# Check the clean data types

presentations_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 22678 entries, 0 to 22677
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   reporting_unit           22678 non-null  string 
 1   reporting_unit_type      22678 non-null  string 
 2   state                    22678 non-null  str    
 3   year                     22678 non-null  string 
 4   triage_category          22678 non-null  string 
 5   number_of_presentations  22062 non-null  float64
 6   small_count_flag         22678 non-null  bool   
dtypes: bool(1), float64(1), str(1), string(4)
memory usage: 1.1 MB


In [320]:
# Save the cleaned presentations data
presentations_output_path = "../data/interim/presentations_clean.csv"

presentations_clean.to_csv(
    presentations_output_path,
    index=False
)

In [321]:
pd.read_csv(presentations_output_path).head()

,reporting_unit,reporting_unit_type,state,year,triage_category,number_of_presentations,small_count_flag
0,National,National,NAT,2011–12,Emergency,647788.0,False
1,National,National,NAT,2012–13,Emergency,714124.0,False
2,National,National,NAT,2013–14,Emergency,785791.0,False
3,National,National,NAT,2014–15,Emergency,843443.0,False
4,National,National,NAT,2015–16,Emergency,899906.0,False


In [322]:
# Standardise columns names and remove extra white space

seen_on_time_clean.columns = (
    seen_on_time_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

categorical_cols = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "peer_group",
    "year",
    "triage_category",
]

for col in categorical_cols:
    seen_on_time_clean[col] = (
        seen_on_time_clean[col]
        .astype("string")
        .str.strip()
    )

seen_on_time_clean["state"] = (
    seen_on_time_clean["state"]
    .map(state_mapping)
)

In [323]:
# Standardise state names
state_mapping = {
    "Vic": "VIC",
    "Qld": "QLD",
    "Tas": "TAS",
}

seen_on_time_clean["state"] = (
    seen_on_time_clean["state"]
    .str.strip()
    .replace(state_mapping)
)

seen_on_time_clean["state"].value_counts(dropna=False)

state
NSW    11499
VIC     3775
QLD     2370
WA      1713
SA      1460
NT       510
TAS      385
ACT      190
NAT       70
Name: count, dtype: int64

In [324]:
# Remove truly empty column (unnamed_7) and rename the caution flag "*"
seen_on_time_clean = seen_on_time_clean.drop(
    columns=["unnamed_7"]
)

seen_on_time_clean = seen_on_time_clean.rename(
    columns={
        "unnamed_9": "seen_on_time_caution_flag"
    }
)

seen_on_time_clean["seen_on_time_caution_flag"] = (
    seen_on_time_clean["seen_on_time_caution_flag"] == "*"
)


In [325]:
seen_on_time_clean["small_count_flag"] = (
    seen_on_time_clean["number_of_presentations"] == "<5"
)

seen_on_time_clean["number_of_presentations"] = pd.to_numeric(
    seen_on_time_clean["number_of_presentations"],
    errors="coerce"
)

In [326]:
seen_on_time_clean.columns

Index(['reporting_unit', 'reporting_unit_type', 'state', 'peer_group', 'year',
       'triage_category', 'number_of_presentations',
       'percentage_of_patients_seen_on_time', 'seen_on_time_caution_flag',
       'peer_group_average', 'small_count_flag'],
      dtype='str')

In [327]:
# Preserve the publication status for percentage_of_patients_seen_on_time and rename the values

def get_publication_status(value):
    if value == "–":
        return "no_patients_reported"
    if value == "NP":
        return "criteria_not_met"
    if value == "NP†":
        return "missing_or_invalid_time_gt_10pct"
    if pd.isna(value):
        return "missing"
    return "reported"


seen_on_time_clean["seen_on_time_status"] = (
    seen_on_time_clean[
        "percentage_of_patients_seen_on_time"
    ].apply(get_publication_status)
)

seen_on_time_clean[
    "percentage_of_patients_seen_on_time"
] = pd.to_numeric(
    seen_on_time_clean[
        "percentage_of_patients_seen_on_time"
    ],
    errors="coerce"
)


In [328]:
# For peer_group_average column, as it can be numeric, not peered or structurally missing, we rename the value

def get_peer_status(row):
    if row["peer_group_average"] == "Not peered":
        return "not_peered"

    if pd.isna(row["peer_group_average"]):
        return "not_applicable"

    return "peered"

seen_on_time_clean["peer_group_status"] = (
    seen_on_time_clean.apply(
        get_peer_status,
        axis=1
    )
)

seen_on_time_clean["peer_group_average"] = pd.to_numeric(
    seen_on_time_clean["peer_group_average"],
    errors="coerce"
)



In [329]:
# Validate the change

print("Rows:", len(seen_on_time_clean))

print(
    "Small-count records:",
    seen_on_time_clean["small_count_flag"].sum()
)

print(
    "Caution-flag records:",
    seen_on_time_clean[
        "seen_on_time_caution_flag"
    ].sum()
)

print(
    seen_on_time_clean[
        "seen_on_time_status"
    ].value_counts(dropna=False)
)

print(
    seen_on_time_clean[
        "peer_group_status"
    ].value_counts(dropna=False)
)

Rows: 21972
Small-count records: 329
Caution-flag records: 385
seen_on_time_status
reported                            20915
missing_or_invalid_time_gt_10pct      562
criteria_not_met                      495
Name: count, dtype: int64
peer_group_status
peered            14420
not_peered         4852
not_applicable     2700
Name: count, dtype: int64


In [330]:
# validate the key

candidate_key = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "peer_group",
    "year",
    "triage_category",
]

assert seen_on_time_clean.duplicated(
    subset=candidate_key
).sum() == 0

In [331]:
# final check the data type
seen_on_time_clean.dtypes

reporting_unit                          string
reporting_unit_type                     string
state                                      str
peer_group                              string
year                                    string
triage_category                         string
number_of_presentations                float64
percentage_of_patients_seen_on_time    float64
seen_on_time_caution_flag                 bool
peer_group_average                     float64
small_count_flag                          bool
seen_on_time_status                        str
peer_group_status                          str
dtype: object

In [332]:
text_cols = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "peer_group",
    "year",
    "triage_category",
    "seen_on_time_status",
    "peer_group_status",
]

for col in text_cols:
    seen_on_time_clean[col] = seen_on_time_clean[col].astype("string")

In [333]:
seen_on_time_clean.dtypes

reporting_unit                          string
reporting_unit_type                     string
state                                   string
peer_group                              string
year                                    string
triage_category                         string
number_of_presentations                float64
percentage_of_patients_seen_on_time    float64
seen_on_time_caution_flag                 bool
peer_group_average                     float64
small_count_flag                          bool
seen_on_time_status                     string
peer_group_status                       string
dtype: object

In [334]:
# save seen_on_time
seen_on_time_output_path = "../data/interim/patients_seen_on_time_clean.csv"

seen_on_time_clean.to_csv(
    seen_on_time_output_path,
    index=False
)

In [335]:
check = pd.read_csv(seen_on_time_output_path)

print(check.shape)
display(check.head())

(21972, 13)


,reporting_unit,reporting_unit_type,state,peer_group,year,triage_category,number_of_presentations,percentage_of_patients_seen_on_time,seen_on_time_caution_flag,peer_group_average,small_count_flag,seen_on_time_status,peer_group_status
0,National,National,NAT,NaN,2011–12,Emergency,644112.0,0.80,False,NaN,False,reported,not_applicable
1,National,National,NAT,NaN,2012–13,Emergency,710966.0,0.82,False,NaN,False,reported,not_applicable
2,National,National,NAT,NaN,2013–14,Emergency,781363.0,0.82,False,NaN,False,reported,not_applicable
3,National,National,NAT,NaN,2014–15,Emergency,839117.0,0.79,False,NaN,False,reported,not_applicable
4,National,National,NAT,NaN,2015–16,Emergency,895191.0,0.77,False,NaN,False,reported,not_applicable


In [336]:
within_4hrs_clean = within_4hrs.copy()

In [337]:
# Clean within_4_hrs_clean

# standardise column names
within_4hrs_clean.columns = (
    within_4hrs_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
categorical_cols = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "peer_group",
    "year",
    "patient_cohort",
]

for col in categorical_cols:
    within_4hrs_clean[col] = (
        within_4hrs_clean[col]
        .astype("string")
        .str.strip()
    )

within_4hrs_clean["state"] = (
    within_4hrs_clean["state"]
    .replace({
        "Vic": "VIC",
        "Qld": "QLD",
        "Tas": "TAS",
    })
)



In [338]:
within_4hrs_clean["state"].value_counts(dropna=False)

state
NSW    19472
VIC     6089
QLD     3792
WA      2752
SA      2336
NT       816
TAS      616
ACT      304
NAT      112
Name: count, dtype: Int64

In [339]:
# Remove the empty column "unnamed_7" and rename the "*" flag column 

within_4hrs_clean = within_4hrs_clean.drop(
    columns=["unnamed_7"]
)

within_4hrs_clean = within_4hrs_clean.rename(
    columns={
        "unnamed_9": "within_4hrs_caution_flag"
    }
)

within_4hrs_clean["within_4hrs_caution_flag"] = (
    within_4hrs_clean["within_4hrs_caution_flag"] == "*"
)

In [340]:
# Handle <5 in the presentation count 
within_4hrs_clean["small_count_flag"] = (
    within_4hrs_clean["number_of_presentations"] == "<5"
)

within_4hrs_clean["number_of_presentations"] = pd.to_numeric(
    within_4hrs_clean["number_of_presentations"],
    errors="coerce"
)

In [341]:
within_4hrs_clean[
    "percentage_who_depart_ed_within_4_hrs"
].value_counts(dropna=False).loc[
    lambda x: x.index.astype(str).isin(["-", "NP", "NP†"])
]

percentage_who_depart_ed_within_4_hrs
NP     1067
-       373
NP†     189
Name: count, dtype: int64

In [342]:
# Preserve the publication status for the performance measure
def get_publication_status(value):
    if pd.isna(value):
        return "missing"

    value = str(value).strip()

    if value == "-":
        return "no_patients_reported"
    if value == "NP":
        return "criteria_not_met"
    if value == "NP†":
        return "missing_or_invalid_time_gt_10pct"

    return "reported"

within_4hrs_clean["within_4hrs_status"] = (
    within_4hrs_clean[
        "percentage_who_depart_ed_within_4_hrs"
    ].apply(get_publication_status)
)

within_4hrs_clean[
    "percentage_who_depart_ed_within_4_hrs"
] = pd.to_numeric(
    within_4hrs_clean[
        "percentage_who_depart_ed_within_4_hrs"
    ],
    errors="coerce"
)
within_4hrs_clean["within_4hrs_status"].value_counts(dropna=False)

within_4hrs_status
reported                            34660
criteria_not_met                     1067
no_patients_reported                  373
missing_or_invalid_time_gt_10pct      189
Name: count, dtype: int64

In [343]:
# Handle the peer-group average 
within_4hrs_clean["peer_group_status"] = (
    within_4hrs_clean.apply(
        get_peer_status,
        axis=1
    )
)

within_4hrs_clean["peer_group_average"] = pd.to_numeric(
    within_4hrs_clean["peer_group_average"],
    errors="coerce"
)

In [344]:
# Standardise text dtypes

text_cols = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "peer_group",
    "year",
    "patient_cohort",
    "within_4hrs_status",
    "peer_group_status",
]

for col in text_cols:
    within_4hrs_clean[col] = (
        within_4hrs_clean[col]
        .astype("string")
    )

In [345]:
# Validation

print("Rows:", len(within_4hrs_clean))

print(
    "Small-count records:",
    within_4hrs_clean["small_count_flag"].sum()
)

print(
    "Caution-flag records:",
    within_4hrs_clean[
        "within_4hrs_caution_flag"
    ].sum()
)

print(
    within_4hrs_clean[
        "within_4hrs_status"
    ].value_counts(dropna=False)
)

print(
    within_4hrs_clean[
        "peer_group_status"
    ].value_counts(dropna=False)
)

Rows: 36289
Small-count records: 653
Caution-flag records: 306
within_4hrs_status
reported                            34660
criteria_not_met                     1067
no_patients_reported                  373
missing_or_invalid_time_gt_10pct      189
Name: count, dtype: Int64
peer_group_status
peered            23200
not_peered         8688
not_applicable     4401
Name: count, dtype: Int64


In [346]:
# Validate the percentage range 
reported_pct = within_4hrs_clean[
    "percentage_who_depart_ed_within_4_hrs"
].dropna()

assert reported_pct.between(0, 1).all()

In [347]:
# Validate the candidate key
candidate_key = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "peer_group",
    "year",
    "patient_cohort",
]

assert within_4hrs_clean.duplicated(
    subset=candidate_key
).sum() == 0

In [348]:
# Save clean within_4_hrs dataset

within_4_hrs_clean_output_path = "../data/interim/time_in_ed_within_4hrs_clean.csv"

within_4hrs_clean.to_csv(
    within_4_hrs_clean_output_path,
    index=False
)

In [349]:
check = pd.read_csv(within_4_hrs_clean_output_path)

print(check.shape)
display(check.head())

(36289, 13)


,reporting_unit,reporting_unit_type,state,peer_group,year,patient_cohort,number_of_presentations,percentage_who_depart_ed_within_4_hrs,within_4hrs_caution_flag,peer_group_average,small_count_flag,within_4hrs_status,peer_group_status
0,National,National,NAT,NaN,2011–12,All patients,6547342.0,0.64,False,NaN,False,reported,not_applicable
1,National,National,NAT,NaN,2012–13,All patients,6717067.0,0.67,False,NaN,False,reported,not_applicable
2,National,National,NAT,NaN,2013–14,All patients,7195903.0,0.73,False,NaN,False,reported,not_applicable
3,National,National,NAT,NaN,2014–15,All patients,7366442.0,0.73,False,NaN,False,reported,not_applicable
4,National,National,NAT,NaN,2015–16,All patients,7465869.0,0.73,False,NaN,False,reported,not_applicable


In [350]:
# Clean time_in_ed

# Standardise column names
time_in_ed_clean.columns = (
    time_in_ed_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

categorical_cols = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "peer_group",
    "year",
    "patient_cohort",
]

for col in categorical_cols:
    time_in_ed_clean[col] = (
        time_in_ed_clean[col]
        .astype("string")
        .str.strip()
    )

# Standardise state abbreviations
time_in_ed_clean["state"] = (
    time_in_ed_clean["state"]
    .replace({
        "Vic": "VIC",
        "Qld": "QLD",
        "Tas": "TAS",
    })
)

In [351]:
time_in_ed_clean["state"].value_counts(dropna=False)

state
NSW    7302
VIC    2286
QLD    1422
WA     1032
SA      876
NT      306
TAS     231
ACT     114
NAT      42
Name: count, dtype: Int64

In [352]:
time_in_ed_clean.columns

Index(['reporting_unit', 'reporting_unit_type', 'state', 'peer_group', 'year',
       'patient_cohort', 'number_of_presentations', 'unnamed_7', 'median_time',
       'unnamed_9', 'time_until_most_patients_90_depart_ed', 'unnamed_11',
       'peer_group_average_90'],
      dtype='str')

In [353]:
# Remove empty column
time_in_ed_clean = time_in_ed_clean.drop(
    columns=["unnamed_7"]
)

In [354]:
# Rename caution-flag columns
time_in_ed_clean = time_in_ed_clean.rename(
    columns={
        "unnamed_9": "median_time_caution_flag",
        "unnamed_11": "p90_time_caution_flag",
    }
)

# Convert "*" markers to boolean flags
time_in_ed_clean["median_time_caution_flag"] = (
    time_in_ed_clean["median_time_caution_flag"]
    .eq("*")
)

time_in_ed_clean["p90_time_caution_flag"] = (
    time_in_ed_clean["p90_time_caution_flag"]
    .eq("*")
)


In [355]:
# Validate flag counts
print(
    "Median-time caution flags:",
    time_in_ed_clean["median_time_caution_flag"].sum()
)

print(
    "P90-time caution flags:",
    time_in_ed_clean["p90_time_caution_flag"].sum()
)

# Check whether the two flags occur on the same rows
flags_match = (
    time_in_ed_clean["median_time_caution_flag"]
    == time_in_ed_clean["p90_time_caution_flag"]
).all()

print("Flags match on all rows:", flags_match)

Median-time caution flags: 94
P90-time caution flags: 94
Flags match on all rows: True


In [356]:
# Handle <5 small count presentation

time_in_ed_clean["small_count_flag"] = (
    time_in_ed_clean["number_of_presentations"] == "<5"
)

time_in_ed_clean["number_of_presentations"] = pd.to_numeric(
    time_in_ed_clean["number_of_presentations"],
    errors="coerce"
)

print(
    "Small-count records:",
    time_in_ed_clean["small_count_flag"].sum()
)

Small-count records: 37


In [357]:
# Handle publication status

def get_publication_status(value):
    if pd.isna(value):
        return "missing"

    value = str(value).strip()

    if value == "-":
        return "no_patients_reported"
    if value == "NP":
        return "criteria_not_met"
    if value == "NP†":
        return "missing_or_invalid_time_gt_10pct"

    return "reported"


time_in_ed_clean["median_time_status"] = (
    time_in_ed_clean["median_time"]
    .apply(get_publication_status)
)

time_in_ed_clean["p90_time_status"] = (
    time_in_ed_clean[
        "time_until_most_patients_90_depart_ed"
    ].apply(get_publication_status)
)




In [358]:
print("Median time status:")
print(
    time_in_ed_clean[
        "median_time_status"
    ].value_counts(dropna=False)
)

print("\nP90 time status:")
print(
    time_in_ed_clean[
        "p90_time_status"
    ].value_counts(dropna=False)
)

Median time status:
median_time_status
reported                            13377
no_patients_reported                  117
criteria_not_met                       68
missing_or_invalid_time_gt_10pct       49
Name: count, dtype: int64

P90 time status:
p90_time_status
reported                            13377
no_patients_reported                  117
criteria_not_met                       68
missing_or_invalid_time_gt_10pct       49
Name: count, dtype: int64


In [359]:
# Duration parsing 


import re
import numpy as np

def duration_to_minutes(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    match = re.fullmatch(
        r"(\d+)\s+hrs?\s+(\d+)\s+mins?",
        value
    )

    if not match:
        return np.nan

    hours = int(match.group(1))
    minutes = int(match.group(2))

    return hours * 60 + minutes

In [ ]:
# Create numeric columns 
time_in_ed_clean["median_time_minutes"] = (
    time_in_ed_clean["median_time"]
    .apply(duration_to_minutes)
)

time_in_ed_clean["p90_time_minutes"] = (
    time_in_ed_clean[
        "time_until_most_patients_90_depart_ed"
    ].apply(duration_to_minutes)
)


In [362]:
# Validate that every value marked as reported was successfully converted 

reported_median = time_in_ed_clean[
    time_in_ed_clean["median_time_status"] == "reported"
]

assert reported_median[
    "median_time_minutes"
].notna().all()


reported_p90 = time_in_ed_clean[
    time_in_ed_clean["p90_time_status"] == "reported"
]

assert reported_p90[
    "p90_time_minutes"
].notna().all()

In [363]:
# Inspect a few rows

time_in_ed_clean[
    [
        "median_time",
        "median_time_minutes",
        "time_until_most_patients_90_depart_ed",
        "p90_time_minutes",
    ]
].head(10)

,median_time,median_time_minutes,time_until_most_patients_90_depart_ed,p90_time_minutes
0,2 hrs 58 mins,178.0,8 hrs 28 mins,508.0
1,2 hrs 53 mins,173.0,7 hrs 56 mins,476.0
2,2 hrs 40 mins,160.0,7 hrs 5 mins,425.0
3,2 hrs 41 mins,161.0,7 hrs 2 mins,422.0
4,2 hrs 44 mins,164.0,6 hrs 53 mins,413.0
5,2 hrs 48 mins,168.0,7 hrs 0 mins,420.0
6,2 hrs 53 mins,173.0,7 hrs 14 mins,434.0
7,2 hrs 58 mins,178.0,7 hrs 29 mins,449.0
8,2 hrs 56 mins,176.0,7 hrs 30 mins,450.0
9,3 hrs 1 mins,181.0,8 hrs 0 mins,480.0


In [364]:
# Validate durations are non-negative 
assert time_in_ed_clean[
    "median_time_minutes"
].dropna().ge(0).all()

assert time_in_ed_clean[
    "p90_time_minutes"
].dropna().ge(0).all()

In [365]:
valid_times = time_in_ed_clean[
    [
        "median_time_minutes",
        "p90_time_minutes"
    ]
].dropna()

(
    valid_times["p90_time_minutes"]
    < valid_times["median_time_minutes"]
).sum()

0

In [ ]:
# Clean peer_group_average_90, as it contains duration strings like 6 hrs 20 mins, Not peered, NaN values for National/LHN rows 

def get_time_peer_status(row):
    if row["peer_group_average_90"] == "Not peered":
        return "not_peered"

    if pd.isna(row["peer_group_average_90"]):
        return "not_applicable"

    return "peered"

time_in_ed_clean["peer_group_status"] = (
    time_in_ed_clean.apply(
        get_time_peer_status,
        axis=1
    )
)


# Convert the actual peer-group benchmark to minute using the existing function duration_to_minutes()

time_in_ed_clean["peer_group_average_90_minutes"] = (
    time_in_ed_clean["peer_group_average_90"]
    .apply(duration_to_minutes)
)

In [367]:
# Validate the status counts

time_in_ed_clean[
    "peer_group_status"
].value_counts(dropna=False)

peer_group_status
peered            10242
not_peered         1716
not_applicable     1653
Name: count, dtype: int64

In [368]:
# Verify that every peered record converted to minutes

peered_rows = time_in_ed_clean[
    time_in_ed_clean["peer_group_status"] == "peered"
]

assert peered_rows[
    "peer_group_average_90_minutes"
].notna().all()

In [369]:
# Check non-applicable / not-peered records do not contains numeric benchmarks
assert time_in_ed_clean.loc[
    time_in_ed_clean["peer_group_status"].isin(
        ["not_peered", "not_applicable"]
    ),
    "peer_group_average_90_minutes"
].isna().all()


In [371]:
# Inspect a few rows

time_in_ed_clean[
    [
        "peer_group",
        "peer_group_average_90",
        "peer_group_average_90_minutes",
        "peer_group_status",
    ]
].tail(20)

,peer_group,peer_group_average_90,peer_group_average_90_minutes,peer_group_status
13591,Major hospitals,12 hrs 5 mins,725.0,peered
13592,Major hospitals,12 hrs 34 mins,754.0,peered
13593,Major hospitals,5 hrs 27 mins,327.0,peered
13594,Major hospitals,5 hrs 40 mins,340.0,peered
13595,Major hospitals,5 hrs 53 mins,353.0,peered
13596,Major hospitals,5 hrs 56 mins,356.0,peered
13597,Major hospitals,6 hrs 15 mins,375.0,peered
13598,Major hospitals,7 hrs 2 mins,422.0,peered
13599,Major hospitals,7 hrs 57 mins,477.0,peered
13600,Major hospitals,8 hrs 11 mins,491.0,peered


In [372]:
time_in_ed_clean[
    [
        "peer_group",
        "peer_group_average_90",
        "peer_group_average_90_minutes",
        "peer_group_status",
    ]
].head(20)

,peer_group,peer_group_average_90,peer_group_average_90_minutes,peer_group_status
0,<NA>,NaN,NaN,not_applicable
1,<NA>,NaN,NaN,not_applicable
2,<NA>,NaN,NaN,not_applicable
3,<NA>,NaN,NaN,not_applicable
4,<NA>,NaN,NaN,not_applicable
5,<NA>,NaN,NaN,not_applicable
6,<NA>,NaN,NaN,not_applicable
7,<NA>,NaN,NaN,not_applicable
8,<NA>,NaN,NaN,not_applicable
9,<NA>,NaN,NaN,not_applicable


In [373]:
# Standardise dtypes in the text columns

text_cols = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "peer_group",
    "year",
    "patient_cohort",
    "median_time_status",
    "p90_time_status",
    "peer_group_status",
]

for col in text_cols:
    time_in_ed_clean[col] = (
        time_in_ed_clean[col]
        .astype("string")
    )

In [375]:
time_in_ed_clean.dtypes

reporting_unit                            string
reporting_unit_type                       string
state                                     string
peer_group                                string
year                                      string
patient_cohort                            string
number_of_presentations                  float64
median_time                                  str
median_time_caution_flag                    bool
time_until_most_patients_90_depart_ed        str
p90_time_caution_flag                       bool
peer_group_average_90                        str
small_count_flag                            bool
median_time_status                        string
p90_time_status                           string
median_time_minutes                      float64
p90_time_minutes                         float64
peer_group_status                         string
peer_group_average_90_minutes            float64
dtype: object

In [376]:
# Validate candidate key
candidate_key = [
    "reporting_unit",
    "reporting_unit_type",
    "state",
    "peer_group",
    "year",
    "patient_cohort",
]

assert time_in_ed_clean.duplicated(
    subset=candidate_key
).sum() == 0

In [377]:
assert time_in_ed_clean.duplicated().sum() == 0

In [379]:
# Final check before saving 

print("Rows:", len(time_in_ed_clean))

print(
    "Small-count records:",
    time_in_ed_clean["small_count_flag"].sum()
)

print(
    "Median caution flags:",
    time_in_ed_clean["median_time_caution_flag"].sum()
)

print(
    "P90 caution flags:",
    time_in_ed_clean["p90_time_caution_flag"].sum()
)

print("\nMedian time status:")
print(
    time_in_ed_clean["median_time_status"]
    .value_counts(dropna=False)
)

print("\nP90 time status:")
print(
    time_in_ed_clean["p90_time_status"]
    .value_counts(dropna=False)
)

print("\nPeer group status:")
print(
    time_in_ed_clean["peer_group_status"]
    .value_counts(dropna=False)
)

Rows: 13611
Small-count records: 37
Median caution flags: 94
P90 caution flags: 94

Median time status:
median_time_status
reported                            13377
no_patients_reported                  117
criteria_not_met                       68
missing_or_invalid_time_gt_10pct       49
Name: count, dtype: Int64

P90 time status:
p90_time_status
reported                            13377
no_patients_reported                  117
criteria_not_met                       68
missing_or_invalid_time_gt_10pct       49
Name: count, dtype: Int64

Peer group status:
peer_group_status
peered            10242
not_peered         1716
not_applicable     1653
Name: count, dtype: Int64


In [380]:
# Save data 

time_in_ed_clean_output_path = (
    "../data/interim/time_in_ed_clean.csv"
)

time_in_ed_clean.to_csv(
    time_in_ed_clean_output_path,
    index=False
)

In [381]:
check = pd.read_csv(time_in_ed_clean_output_path)

print(check.shape)
display(check.head())

(13611, 19)


,reporting_unit,reporting_unit_type,state,peer_group,year,patient_cohort,number_of_presentations,median_time,median_time_caution_flag,time_until_most_patients_90_depart_ed,p90_time_caution_flag,peer_group_average_90,small_count_flag,median_time_status,p90_time_status,median_time_minutes,p90_time_minutes,peer_group_status,peer_group_average_90_minutes
0,National,National,NAT,NaN,2011–12,All patients,6547342.0,2 hrs 58 mins,False,8 hrs 28 mins,False,NaN,False,reported,reported,178.0,508.0,not_applicable,NaN
1,National,National,NAT,NaN,2012–13,All patients,6717067.0,2 hrs 53 mins,False,7 hrs 56 mins,False,NaN,False,reported,reported,173.0,476.0,not_applicable,NaN
2,National,National,NAT,NaN,2013–14,All patients,7195903.0,2 hrs 40 mins,False,7 hrs 5 mins,False,NaN,False,reported,reported,160.0,425.0,not_applicable,NaN
3,National,National,NAT,NaN,2014–15,All patients,7366442.0,2 hrs 41 mins,False,7 hrs 2 mins,False,NaN,False,reported,reported,161.0,422.0,not_applicable,NaN
4,National,National,NAT,NaN,2015–16,All patients,7465869.0,2 hrs 44 mins,False,6 hrs 53 mins,False,NaN,False,reported,reported,164.0,413.0,not_applicable,NaN


## Cleaning Summary

Four emergency department tables were cleaned and standardised:

- `presentations_clean`
- `patients_seen_on_time_clean`
- `time_in_ed_within_4hrs_clean`
- `time_in_ed_clean`

Key transformations included:

- Standardising column names and state abbreviations
- Removing structurally empty columns
- Preserving small-count suppression (`<5`) through boolean flags
- Preserving publication-status codes (`NP`, `NP†`, `_`) as separate status fields
- Preserving data-quality caution flags (`*`)
- Distinguishing `not_peered` from structurally `not_applicable` peer-group values
- Converting reported percentages to numeric values
- Converting ED duration strings into total minutes
- Validating candidate-key uniqueness and row counts after cleaning

The original source workbook remains unchanged, and cleaned outputs are stored in `data/interim/`.

In [382]:
reporting_units = pd.concat([
    presentations_clean[
        ["reporting_unit", "reporting_unit_type", "state"]
    ],
    seen_on_time_clean[
        ["reporting_unit", "reporting_unit_type", "state"]
    ],
    within_4hrs_clean[
        ["reporting_unit", "reporting_unit_type", "state"]
    ],
    time_in_ed_clean[
        ["reporting_unit", "reporting_unit_type", "state"]
    ]
]).drop_duplicates()

In [383]:
state_check = (
    reporting_units
    .groupby(
        ["reporting_unit", "reporting_unit_type"]
    )["state"]
    .nunique()
)

state_check[state_check > 1]

Series([], Name: state, dtype: int64)

In [384]:
reporting_units.shape

(402, 3)

In [385]:
reporting_units.head()

,reporting_unit,reporting_unit_type,state
0,National,National,NAT
70,Albury Wodonga Health [Albury Campus],Hospital,NSW
140,Armidale Hospital,Hospital,NSW
210,Auburn Hospital,Hospital,NSW
280,Ballina District Hospital,Hospital,NSW
